In [6]:
import asyncio
import re
from typing import List, Dict
from google.oauth2 import service_account
from googleapiclient.discovery import build
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:

RAW_SHEET_NAME = "Sheet1" 
PROCESSED_SHEET_NAME = "Sheet2"  
CONTENT_DELIMITER = "-----NEXT CONTENT FROM HERE-----"
CATEGORIES = ["ABOUT_US", "EBOOK", "COURSES", "RECENT_BLOG", "TESTIMONIALS", "WEBINAR", "SERVICES", "PODCAST", "SHOP"]
COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "M", "EBOOK": "N", "COURSES": "O", "RECENT_BLOG": "P",
    "TESTIMONIALS": "Q", "WEBINAR": "R", "SERVICES": "S", "PODCAST": "T", "SHOP": "U"
}
EXTRACTION_METADATA_COLUMN = "V"
CATEGORY_THRESHOLDS = {
    "ABOUT_US": 200, "EBOOK": 200, "COURSES": 300, "RECENT_BLOG": 450, 
    "TESTIMONIALS": 100, "WEBINAR": 150, "SERVICES": 150, "PODCAST": 200, "SHOP": 100
}
OPENAI_API_KEY = ""
SHEET_URL = "https://docs.google.com/spreadsheets/d/1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc/edit?gid=0#gid=0"
CREDENTIALS_FILE = "url-to-email-445616-cebe4868914f.json"  
PROMPT_TEMPLATES = {
    "RECENT_BLOG": """
You are provided with multiple blog posts. Your task is to evaluate each post based on the following criteria and select the one with the highest total score. Then, output **only** the content text of the selected blog post. Do not include any explanations, scores, rankings, metadata, or additional information.

**Evaluation Criteria:**
1. *Recency*: Score based on how recently the post was published, using the publication date provided.
2. *Relevance to Prospect's Industry*: Score based on how relevant the content is to the specified industry or keywords, using the excerpt provided.
3. *Post Length*: Score based on the word count, with a minimum of 200 words.

**Scoring Guidelines**:
- *Recency*:
  - Within the last month: 10
  - 1-3 months ago: 8
  - 3-6 months ago: 6
  - 6-12 months ago: 4
  - Over 12 months ago: 2
- *Relevance to Prospect's Industry*:
  - Highly relevant (multiple keywords or strong topic alignment): 10
  - Moderately relevant (some keywords or partial alignment): 7
  - Slightly relevant (few keywords or weak alignment): 5
  - Not relevant: 0
- *Post Length*:
  - Less than 200 words: 0
  - 200-500 words: 5
  - 500-1000 words: 7
  - 1000-2000 words: 9
  - 2000+ words: 10

**Final Output**:
- Output **only** the content text of the blog post with the highest total score.
- Do **not** include any explanations, scores, rankings, metadata (e.g., "Content length", "Line count", "Word Count"), or any other information.
- Do **not** mention other posts or provide any additional context.

{content}
"""
}

In [8]:
# GoogleSheetsManager class for API interactions
class GoogleSheetsManager:
    def __init__(self, credentials_file):
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=creds)

    def extract_spreadsheet_id(self, sheet_url):
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")

# Parse content into pieces based on delimiter and metadata
def parse_content_pieces(content, expected_count, category, row_num):
    if not content or not content.strip():
        return []
    cleaned_content = content.strip()
    delimiters_to_try = [
        CONTENT_DELIMITER,
        CONTENT_DELIMITER.strip(),
        "-----NEXT CONTENT FROM HERE-----",
        "--- --NEXT CONTENT FROM HERE-----",
        "-----NEXT CONTENT FROM HERE--- --",
    ]
    pieces = None
    delimiter_used = None
    for delimiter in delimiters_to_try:
        if delimiter in cleaned_content:
            pieces = cleaned_content.split(delimiter)
            delimiter_used = delimiter
            break
    if pieces is None:
        pieces = [cleaned_content]
    else:
        pass
    cleaned_pieces = [piece.strip() for piece in pieces if piece.strip()]
    if expected_count == 1 and len(cleaned_pieces) == 1:
        return cleaned_pieces
    elif expected_count > 1:
        if len(cleaned_pieces) >= expected_count:
            return cleaned_pieces[:expected_count]
        else:
            return cleaned_pieces
    else:
        return cleaned_pieces

# Read and organize data from the input sheet
async def read_data_from_sheet(spreadsheet_id, sheet_mgr, categories):
    category_columns = {cat: COLUMN_TO_WRITE_URL_TO.get(cat.upper()) for cat in categories}
    if any(col is None for col in category_columns.values()):
        raise ValueError("One or more categories do not have a defined column in COLUMN_TO_WRITE_URL_TO")
    metadata_column = EXTRACTION_METADATA_COLUMN
    ranges = [f"{RAW_SHEET_NAME}!{col}2:{col}" for col in category_columns.values()] + [f"{RAW_SHEET_NAME}!{metadata_column}2:{metadata_column}"]
    try:
        response = await asyncio.to_thread(
            sheet_mgr.service.spreadsheets().values().batchGet(spreadsheetId=spreadsheet_id, ranges=ranges).execute
        )
    except Exception as e:
        raise RuntimeError(f"Failed to read data from spreadsheet {spreadsheet_id}: {str(e)}")
    value_ranges = response.get('valueRanges', [])
    column_data = {}
    for i, col in enumerate(category_columns.values()):
        column_data[col] = value_ranges[i].get('values', [])
    column_data[metadata_column] = value_ranges[-1].get('values', [])
    num_rows = max(len(values) for values in column_data.values()) if column_data else 0
    data = []
    for i in range(num_rows):
        row_data = {}
        metadata_value = column_data[metadata_column][i][0] if i < len(column_data[metadata_column]) and column_data[metadata_column][i] else ''
        category_n_dict = {}
        for part in metadata_value.split(','):
            if '=' in part:
                cat, n_str = part.split('=', 1)
                try:
                    n = int(n_str.strip())
                    category_n_dict[cat.strip().upper()] = n
                except ValueError:
                    pass
        for category in categories:
            category_upper = category.upper()
            if category_upper in category_n_dict and category_n_dict[category_upper] > 0:
                col = category_columns[category]
                content = column_data[col][i][0] if i < len(column_data[col]) and column_data[col][i] else ''
                expected_count = category_n_dict[category_upper]
                parsed_pieces = parse_content_pieces(content, expected_count, category, i+2)
                if parsed_pieces:
                    row_data[category] = parsed_pieces
        data.append(row_data)
    return data

# Format RECENT_BLOG content for AI processing
def explore_all_content1(data, row_number, category):
    try:
        row_data = data[row_number - 2]
        if category in row_data:
            pieces = row_data[category]
            output = []
            for idx, content in enumerate(pieces, start=1):
                output.append(f"*Blog Post {idx}*:")
                output.append(content)
                output.append("")
                output.append(f"Content length: {len(content)} characters")
                output.append(f"Line count: {len(content.splitlines())}")
                output.append(f"Word Count: {len(content.split())}")
            return "\n".join(output)
        else:
            return f"Error: Category '{category}' not found in row {row_number}."
    except IndexError:
        return f"Error: Row {row_number} not found. Available rows: 2 to {len(data) + 1}"

# Set up LangChain pipeline for AI processing
def create_chain(category):
    if category not in PROMPT_TEMPLATES:
        raise ValueError(f"No prompt defined for category {category}")
    prompt = PromptTemplate(
        input_variables=["content"],
        template=PROMPT_TEMPLATES[category]
    )
    llm = ChatOpenAI(model="gpt-4", api_key=OPENAI_API_KEY)
    return prompt | llm | StrOutputParser()

# Process RECENT_BLOG row with AI
async def process_row_with_langchain(content_output, category):
    chain = create_chain(category)
    if not content_output.strip() or "Error:" in content_output:
        return f"No valid content provided for {category}"
    try:
        response = await chain.ainvoke({"content": content_output})
        return response
    except Exception as e:
        return f"Error processing row for {category}"

# Process a category column according to the algorithm
async def process_category(data, category, sheet_mgr, spreadsheet_id):
    column = COLUMN_TO_WRITE_URL_TO[category.upper()]
    values = []
    for row_idx in range(2, len(data) + 2):
        row_data = data[row_idx - 2]
        pieces = row_data.get(category, [])
        if not pieces:
            output = "no content"
        elif len(pieces) == 1:
            output = pieces[0]
        else:
            if category == "RECENT_BLOG":
                content_output = explore_all_content1(data, row_idx, category)
                if "Error:" in content_output:
                    output = "No valid content provided"
                else:
                    output = await process_row_with_langchain(content_output, category)
            else:
                threshold = CATEGORY_THRESHOLDS.get(category.upper(), 0)
                candidates = [p for p in pieces if len(p.split()) >= threshold]
                if candidates:
                    output = max(candidates, key=lambda p: len(p.split()))
                else:
                    output = max(pieces, key=lambda p: len(p.split()))
        values.append([output])
    range_name = f"{PROCESSED_SHEET_NAME}!{column}2:{column}{len(data) + 1}"
    try:
        sheet_mgr.service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=range_name,
            valueInputOption='RAW',
            body={'values': values}
        ).execute()
    except Exception as e:
        pass

# Main function to orchestrate the pipeline
async def main():
    try:
        sheet_mgr = GoogleSheetsManager(CREDENTIALS_FILE)
        spreadsheet_id = sheet_mgr.extract_spreadsheet_id(SHEET_URL)
        data = await read_data_from_sheet(spreadsheet_id, sheet_mgr, CATEGORIES)
        for category in CATEGORIES:
            await process_category(data, category, sheet_mgr, spreadsheet_id)
    except Exception as e:
        pass

# Run the pipeline
# if __name__ == "__main__":
#     asyncio.run(main())

await main()

House Passes Sweeping Tax Bill: SALT Cap Expansion, TCJA Extensions, and Other Tax Provisions Head to Senate
May 23, 2025
|
Tax Changes
On May 22, 2025, the House of Representatives passed H.R. 1, sending a broad tax and domestic policy package to the Senate for further consideration. While the bill represents a significant step in advancing President Trump’s proposed agenda, none of its provisions are currently law, and the final version remains subject to change.
Trump’s Tax Proposal: What It Means for Businesses and Individuals
May 21, 2025
|
Tax Changes
As of May 21, 2025, President Trump’s proposed tax legislation remains under active debate and revision. Introduced as part of a broader budget reconciliation package, the proposal seeks to extend or expand key provisions from the Tax Cuts and Jobs Act (TCJA) while...
Treasury Suspends Penalties and FinCEN Delays BOI Reporting Deadline
Mar 3, 2025
|
News
On March 2, 2025, the U.S. Treasury Department announced that it would not impo